# Chatterbox ONNX Q4 — telefonda ishlash tezligini baholash

**Maqsad:** Chatterbox'ni telefonda (serversiz) ishlatish mumkinmi, hal qilish. Bu daftar ko'p tilli Q4 ONNX modelining (≈790 MB) CPU'dagi tezligini o'lchaydi. Sizning o'zbekcha fine-tune ham xuddi shu arxitekturada (0.5B Llama + dekoder), shuning uchun tezlik taxminan bir xil bo'ladi.

**Qanday ishlatish:** Runtime → Change runtime type → **CPU** (GPU tanlamang!) → Runtime → Run all. Oxirgi katakcha natijasini menga yuboring.

Eslatma: Colab CPU telefon protsessoriga to'liq teng emas, natija taxminiy. Tasodifiy kirishlar ishlatiladi, shuning uchun audio sifati emas, faqat **tezlik** o'lchanadi.

In [ ]:
!pip -q install onnxruntime numpy huggingface_hub


In [ ]:
import time, os, numpy as np, onnxruntime as ort
from huggingface_hub import hf_hub_download, list_repo_files

REPOS = ["onnx-community/chatterbox-multilingual-ONNX", "Vocabook/chatterbox-multilingual-ONNX-q4"]
LOCAL = "/content/cb_onnx"
NAMES = ["speech_encoder", "embed_tokens", "language_model", "conditional_decoder"]

def pick(files, name):
    cand = [f for f in files if f.endswith(".onnx") and os.path.basename(f).startswith(name)]
    for key in ("q4", "int8", "quant", "fp16", ""):
        for f in cand:
            if key in os.path.basename(f):
                return f
    return None

def fetch(repo):
    files = list_repo_files(repo)
    print(repo, "->", len(files), "fayl")
    for f in files:
        if f.endswith((".onnx", ".onnx_data")):
            print("   ", f)
    chosen = {n: pick(files, n) for n in NAMES}
    print("tanlangan:", chosen)
    if any(v is None for v in chosen.values()):
        return None
    out = {}
    for n, f in chosen.items():
        # local_dir: haqiqiy fayllar (symlink emas) — ONNX Runtime tashqi ma'lumot faylini shunda topadi
        out[n] = hf_hub_download(repo, f, local_dir=LOCAL)
        if f + "_data" in files:
            hf_hub_download(repo, f + "_data", local_dir=LOCAL)
    return out

paths = None
for repo in REPOS:
    try:
        paths = fetch(repo)
    except Exception as e:
        print("xato:", repo, e)
    if paths:
        break
assert paths, "Model fayllari topilmadi — yuqoridagi ro'yxatni menga yuboring"

for n, p in paths.items():
    print(f"{n:22s} {os.path.getsize(p)/1e6:8.1f} MB")

def load(threads):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    return {n: ort.InferenceSession(p, so, providers=["CPUExecutionProvider"]) for n, p in paths.items()}

S = load(2)
for n, s in S.items():
    ins = s.get_inputs()
    print("\n", n, "kirishlar:", [(i.name, i.shape) for i in ins[:5]], "... jami", len(ins))
    print("   chiqishlar:", [o.name for o in s.get_outputs()[:4]], "... jami", len(s.get_outputs()))


In [ ]:
def run(sess, feeds):
    names = {i.name for i in sess.get_inputs()}
    return sess.run(None, {k: v for k, v in feeds.items() if k in names})

def bench(threads, text_len=60, gen_tokens=100):
    S = load(threads)
    se, et, lm, cd = S["speech_encoder"], S["embed_tokens"], S["language_model"], S["conditional_decoder"]

    audio = (np.random.randn(1, 24000 * 6) * 0.05).astype(np.float32)
    t = time.time()
    cond_emb, prompt_token, ref_x, prompt_feat = run(se, {"audio_values": audio})[:4]
    t_enc = time.time() - t

    ids = np.random.randint(10, 500, (1, text_len)).astype(np.int64)
    pos = (np.arange(text_len)[None, :] - 1).astype(np.int64)
    exg = np.array([0.5], np.float32)
    emb = run(et, {"input_ids": ids, "position_ids": pos, "exaggeration": exg})[0]
    x = np.concatenate([cond_emb, emb], 1)

    past_in = [i for i in lm.get_inputs() if i.name.startswith("past_key_values")]
    H = past_in[0].shape[1] if isinstance(past_in[0].shape[1], int) else 16
    D = past_in[0].shape[3] if isinstance(past_in[0].shape[3], int) else 64
    names = [i.name for i in past_in]
    past = {n: np.zeros((1, H, 0, D), np.float32) for n in names}
    mask = np.ones((1, x.shape[1]), np.int64)

    t = time.time()
    out = run(lm, dict(inputs_embeds=x, attention_mask=mask, **past))
    t_prefill = time.time() - t
    past = dict(zip(names, out[1:]))

    steps = []
    for i in range(gen_tokens):
        tok = np.random.randint(0, 6561, (1, 1)).astype(np.int64)
        t = time.time()
        e = run(et, {"input_ids": tok, "position_ids": np.array([[i + 1]], np.int64), "exaggeration": exg})[0]
        mask = np.concatenate([mask, np.ones((1, 1), np.int64)], 1)
        out = run(lm, dict(inputs_embeds=e, attention_mask=mask, **past))
        past = dict(zip(names, out[1:]))
        steps.append(time.time() - t)

    toks = np.random.randint(0, 6561, (1, gen_tokens)).astype(np.int64)
    speech = np.concatenate([prompt_token, toks], 1)
    t = time.time()
    run(cd, {"speech_tokens": speech, "speaker_embeddings": ref_x, "speaker_features": prompt_feat})
    t_dec = time.time() - t

    audio_sec = gen_tokens / 25.0          # S3 tokenlari ~25 Hz
    total = t_prefill + sum(steps) + t_dec  # ovoz kodlagich har gap uchun emas, namuna uchun bir marta
    print(f"--- {threads} oqim ---")
    print(f"ovoz kodlagich (namuna uchun 1 marta): {t_enc:.1f} s")
    print(f"prefill: {t_prefill:.2f} s | token boshiga: {1000*np.mean(steps):.0f} ms | dekoder: {t_dec:.1f} s")
    print(f"{audio_sec:.1f} s audio uchun jami {total:.1f} s  =>  RTF = {total/audio_sec:.2f} (1.0 = real vaqt)")
    print(f"3 daqiqalik video (~180 s nutq) uchun taxminan {total/audio_sec*180/60:.0f} daqiqa (CFG'siz)\n")

for th in (2, 4):
    bench(th)


## Natijani qanday o'qish

- **RTF ≤ 3:** telefonda amaliy (dublaj fon rejimida bir necha o'n daqiqa).
- **RTF 3–6:** faqat qisqa videolar uchun.
- **RTF > 6:** telefonda mantiqsiz, offline ovoz uchun VITS'ni yaxshilaymiz.

CFG (cfg_weight) yoqilsa til modeli ≈2 barobar sekinlashadi; bu o'lchovda CFG yo'q.